In [0]:
catalog = "movie_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Célula para validação
O script abaixo é utilizado para validar a qualidade das transformações executadas nessa etapa.

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

# 1. movies_info
- Tradução geral das colunas
- Limpeza e tradução das categorias de "status"
- Deduplicação com base na data de ingestão
- Tratamento de data multi-formato
- Nova coluna "ano_lancamento" a partir da coluna "release_date"

In [0]:
from pyspark.sql.window import Window

df_bronze_tb_movies_info = spark.table(f"{bronze_schema}.tb_movies_info")

window_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_timestamp").desc()) # Deduplicação com base na data de ingestão

formatos_data = [ # Criado para tratamento de datas multi-formato
    "yyyy-MM-dd'T'HH:mm:ss.SSSXXX",    # ISO 8601 completo com fuso
    "yyyy-MM-dd'T'HH:mm:ss",           # ISO 8601 simples
    "yyyy-MM-dd HH:mm:ss",             # Timestamp padrão
    "yyyy-MM-dd",                      # ISO data simples (Parece ser o mais comum nos dados)
    "dd/MM/yyyy",                      # Padrão Brasil com barra
    "dd-MM-yyyy",                      # Padrão Brasil com hífen
    "MM/dd/yyyy",                      # Padrão US com barra
    "yyyy/MM/dd",                      # Padrão ano/mês/dia
    "dd.MM.yyyy",                      # Padrão com ponto
    "yyyyMMdd"                         # Compacto
]

df_silver_tb_info_filmes = (
    df_bronze_tb_movies_info
    # 1. Deduplicação com base na data de ingestão
    .withColumn("_row_num", F.row_number().over(window_dedup))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    # 2. Limpeza e tradução do status
    .withColumn("status_pad", F.trim(F.upper(F.col("status"))))
    .withColumn("status_filme",
        F.when(F.col("status_pad") == "RELEASED", "Lançado")
         .when(F.col("status_pad") == "POST PRODUCTION", "Pós-Produção")
         .when(F.col("status_pad") == "IN PRODUCTION", "Em Produção")
         .when(F.col("status_pad") == "PLANNED", "Planejado")
         .when(F.col("status_pad") == "RUMORED", "Rumores")
         .when(F.col("status_pad") == "CANCELED", "Cancelado")
         .otherwise("Não Informado"))
    # 3. Tratamento de data multi-formato
    .withColumn("release_date", F.trim(F.col("release_date")))
    .withColumn("release_date", F.coalesce(*([F.try_to_date("release_date", fmt) for fmt in formatos_data])))
    # 4. Criação de nova coluna "ano_lancamento"
    .withColumn("ano_lancamento", F.year(F.col("release_date")))
    # 5. Seleção e tradução das colunas finais
    .select("id", "title", "original_title", "release_date", "runtime", "original_language", "status_filme", "overview", "tagline")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

dq_check("silver.tb_info_filmes", "id não nulo", df_silver_tb_info_filmes, F.col("id_filme").isNotNull())
dq_check_unique("silver.tb_info_filmes", "id único", df_silver_tb_info_filmes, ["id_filme"])

df_silver_tb_info_filmes.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_info_filmes")

# 2. movies_financials
- Tratamento de valores textuais que representam ausência de dados (substituir por NULL)
- Higienização das colunas de orçamento e receita
- Conversão das métricas para o tipo adequado e garantir que valores zerados/negativos sejam tratados como ausentes
- Calcular os valores equivalentes em reais (BRL) aplicando a taxa de cotação obtida
- Derivação das colunas de Lucro (Dólar/Real) e Margem de Lucro Percentual, garantindo que operações aritméticas com valores ausentes não invalidem o resultado e evitando divisões por zero

In [0]:
from pyspark.sql.types import DecimalType

df_bronze_tb_movies_financial = spark.table(f"{bronze_schema}.tb_movies_financial")

ausentes = ["Unknown", "Não Informado", "N/A", "n/a", "null", "NULL", "NaN", "", "-", "None", "undefined"]

df_silver_tb_financeiro_filmes = (
    df_bronze_tb_movies_financial
    # 1. Tratamento dos valores textuais que representam ausência de dados
    .withColumn("revenue", F.when(F.trim(F.col("revenue")).isin(ausentes), "NULL")
     .otherwise(F.col("revenue")))
    .withColumn("budget", F.when(F.trim(F.col("budget")).isin(ausentes), "NULL")
     .otherwise(F.col("budget")))
    # 2. Higienzação das colunas de orçamento e receita
    .withColumn("revenue", F.regexp_replace("revenue", r"[^\d]", ""))
    .withColumn("budget", F.regexp_replace("budget", r"[^\d]", ""))
    # 3. Conversão das métricas para o tipo adequado
    .withColumn("revenue", F.col("revenue").cast(DecimalType(18, 2)))
    .withColumn("budget", F.col("budget").cast(DecimalType(18, 2)))
    # 4. Tradução das colunas "id", "revenue" e "budget"
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("revenue", "receita_usd")
    .withColumnRenamed("budget", "orcamento_usd")
    # 5. Convertendo os valores para BRL
)

# 3. movies_metrics
- Limpeza na coluna "popularity"
- Conversão de tipagem para valores decimais e inteiros nas colunas "num_Votes" e "averageRating"
- Tratamento de inconsistências em escala e limites numéricos

In [0]:
df_bronze_tb_movies_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_silver_tb_metricas_engajamento = (
    df_bronze_tb_movies_metrics
    # 1. Conversão de tipagem segura na coluna ""popularity""
    
    # 2. 

)